<a href="https://colab.research.google.com/github/kamilsielski6/spark/blob/main/Logistic_Regression_Titanic.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 1. Wczytanie danych

In [2]:
from pyspark.sql import SparkSession
from pyspark.ml.classification import LogisticRegression

spark = SparkSession.builder.appName('titanic_logreg').getOrCreate()
df = spark.read.csv('titanic.csv', inferSchema = True, header = True)
df.show(3)

+--------+------+--------------------+------+----+-----------------------+-----------------------+-------+
|Survived|Pclass|                Name|   Sex| Age|Siblings/Spouses Aboard|Parents/Children Aboard|   Fare|
+--------+------+--------------------+------+----+-----------------------+-----------------------+-------+
|       0|     3|Mr. Owen Harris B...|  male|22.0|                      1|                      0|   7.25|
|       1|     1|Mrs. John Bradley...|female|38.0|                      1|                      0|71.2833|
|       1|     3|Miss. Laina Heikk...|female|26.0|                      0|                      0|  7.925|
+--------+------+--------------------+------+----+-----------------------+-----------------------+-------+
only showing top 3 rows


In [3]:
df.printSchema() # sprawdzam schemat danych

root
 |-- Survived: integer (nullable = true)
 |-- Pclass: integer (nullable = true)
 |-- Name: string (nullable = true)
 |-- Sex: string (nullable = true)
 |-- Age: double (nullable = true)
 |-- Siblings/Spouses Aboard: integer (nullable = true)
 |-- Parents/Children Aboard: integer (nullable = true)
 |-- Fare: double (nullable = true)



In [4]:
df.columns # lista kolumn

['Survived',
 'Pclass',
 'Name',
 'Sex',
 'Age',
 'Siblings/Spouses Aboard',
 'Parents/Children Aboard',
 'Fare']

In [6]:
my_col = df.select(['Survived','Pclass','Sex','Age','Siblings/Spouses Aboard','Parents/Children Aboard','Fare'])

In [7]:
final_data = my_col.na.drop()

In [10]:
from pyspark.ml.feature import (VectorAssembler, StringIndexer, VectorIndexer, OneHotEncoder)
# Kodowanie płci (StringIndexer + OneHotEncoder)
gender_indexer = StringIndexer(inputCol = 'Sex', outputCol = 'SexIndex')
gender_encoder = OneHotEncoder(inputCol='SexIndex', outputCol = 'SexVec')

In [9]:
# Kodowanie portu zaokrętowania
embark_indexer = StringIndexer(inputCol = 'Embarked', outputCol = 'EmbarkIndex')
embark_encoder = OneHotEncoder(inputCol = 'EmbarkIndex', outputCol = 'EmbarkVec')

In [11]:
# Scalenie cech w jeden wektor (VectorAssembler)
assembler = VectorAssembler(inputCols = ['Pclass', 'SexVec', 'Age', 'SibSp', 'Parch', 'Fare', 'EmbarkVec'], outputCol = 'features')

--Spark ML wymaga jednej kolumny-wektora z cechami. VectorAssembler łączy cechy numeryczne (Pclass, Age, SibSp, Parch, Fare) i zakodowane kategorie (SexVec, EmbarkVec) w kolumnę features. Zwróć uwagę: bierze zakodowane wersje (SexVec/EmbarkVec), nie surowe Sex/Embarked.

In [12]:
# Definiuje model

from pyspark.ml import Pipeline

log_reg = LogisticRegression(featuresCol = 'features', labelCol = 'Survived')

In [14]:
# Złożenie pipeline'u
pipeline = Pipeline(stages = [gender_indexer, embark_indexer,
                             gender_encoder, embark_encoder,
                             assembler, log_reg])



""" Łączy wszystkie etapy w jeden łańcuch wykonywany po kolei. Kolejność jest istotna — najpierw indeksery,
potem enkodery (bo enkoder potrzebuje indeksu), potem assembler (potrzebuje zakodowanych kolumn), a na końcu model.
Pipeline gwarantuje, że trening i predykcja przechodzą dokładnie te same kroki, bez wycieku danych. """

In [16]:
#  Podział na trening i test
train, test = final_data.randomSplit([0.7, 0.3], seed=42)

In [18]:
from pyspark.ml.feature import (VectorAssembler, StringIndexer, OneHotEncoder)
from pyspark.ml import Pipeline
from pyspark.ml.classification import LogisticRegression

# Redefine the gender indexer and encoder
gender_indexer = StringIndexer(inputCol = 'Sex', outputCol = 'SexIndex')
gender_encoder = OneHotEncoder(inputCol='SexIndex', outputCol = 'SexVec')

# The 'Embarked' column was not found in the DataFrame, so its indexer and encoder are removed.

# Redefine the assembler with correct column names and without 'EmbarkVec'
assembler = VectorAssembler(inputCols = ['Pclass', 'SexVec', 'Age', 'Siblings/Spouses Aboard', 'Parents/Children Aboard', 'Fare'], outputCol = 'features')

# Redefine the Logistic Regression model
log_reg = LogisticRegression(featuresCol = 'features', labelCol = 'Survived')

# Reconstruct the pipeline without the 'Embarked' related stages
pipeline = Pipeline(stages = [gender_indexer, gender_encoder, assembler, log_reg])

# trening modelu
fit_model = pipeline.fit(train)

Uruchamia cały pipeline na danych treningowych: dopasowuje indeksery (uczą się list kategorii), enkodery, a na końcu trenuje regresję logistyczną. Zwraca gotowy PipelineModel.


In [19]:
# predykcja na zbiorze testowym
results = fit_model.transform(test)

In [21]:
# podgląd predykcji na trzech wierszach

results.select('prediction', 'Survived').show(3)

+----------+--------+
|prediction|Survived|
+----------+--------+
|       1.0|       0|
|       1.0|       0|
|       1.0|       0|
+----------+--------+
only showing top 3 rows


In [23]:
# ocena modelu AUC (pole pod krzywą ROC)

from pyspark.ml.evaluation import BinaryClassificationEvaluator

eval = BinaryClassificationEvaluator(rawPredictionCol = 'rawPrediction', labelCol = 'Survived')
AUC = eval.evaluate(results)
AUC

# Wynik ~0,85 oznacza dobrą zdolność modelu do odróżniania ocalałych od nieocalałych.

0.8654792129482707